# TPC-H Orders

**Dataset:** `samples.tpch.orders`

**Difficulty:** Easy

**Topics:** filter, date functions, groupBy, aggregation

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Date Functions

| Function | What it does |
|----------|-------------|
| `F.year(col)` | Extracts year from a date/timestamp |
| `F.month(col)` | Extracts month number (1–12) |
| `F.dayofweek(col)` | Day of week (1=Sunday … 7=Saturday) |
| `F.to_date(col, fmt)` | Parses a string as a date |
| `F.datediff(end, start)` | Number of days between two dates |
| `df.filter(F.year("date_col") == 1995)` | Filter by extracted date part |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.tpch.orders")

# Extract date parts and count orders per year
df.select(
    F.year("o_orderdate").alias("yr"),
    F.month("o_orderdate").alias("mo")
).groupBy("yr").count().orderBy("yr").show()

# How many distinct order statuses exist?
df.select("o_orderstatus").distinct().show()

In [0]:
display(df)

## Problem 1

Count the number of orders for each **order status**. The `o_orderstatus`
column uses single-character codes: `O` (Open), `F` (Fulfilled), `P` (Partial).
Load `samples.tpch.orders`.

**Expected output columns:**
- `o_orderstatus` - the status code
- `order_count` - number of orders with that status

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1 
result_1 = df.groupBy("o_orderstatus").agg(F.count("o_orderkey").alias("order_count"))

display(result_1) # replace this

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'o_orderstatus' in cols, "Missing column: o_orderstatus"
assert 'order_count' in cols, "Missing column: order_count"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
statuses = set(r['o_orderstatus'] for r in result_1.collect())
assert statuses.issubset({'O', 'F', 'P'}), f"Unexpected status values: {statuses - {'O', 'F', 'P'}}"
assert all(r['order_count'] > 0 for r in result_1.collect()), "All order counts must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Calculate the **total revenue per order priority** level.
Sum `o_totalprice` grouped by `o_orderpriority` and sort descending by revenue.

**Expected output columns:**
- `o_orderpriority` - priority label
- `total_revenue` - sum of `o_totalprice` for that priority (sorted descending)

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2
result_2 = df.groupBy("o_orderpriority").agg(F.sum("o_totalprice").alias("total_revenue")).orderBy(F.desc("total_revenue"))
display(result_2)  # replace this

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'o_orderpriority' in cols, "Missing column: o_orderpriority"
assert 'total_revenue' in cols, "Missing column: total_revenue"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
revenues = [float(r['total_revenue']) for r in result_2.collect()]
assert revenues == sorted(revenues, reverse=True), "Results must be sorted by total_revenue descending"
assert all(r > 0 for r in revenues), "All total_revenue values must be positive"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Count the total number of orders placed in **year 1995**.
Use `F.year()` to extract the year from `o_orderdate` and filter.
Return a single-row DataFrame.

**Expected output columns:**
- `order_count_1995` - total number of orders with an order date in 1995

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3 
result_3 = df.filter(F.year("o_orderdate") == 1995).agg(F.count("o_orderkey").alias("order_count_1995"))

display(result_3) # replace this

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'order_count_1995' in cols, "Missing column: order_count_1995"
cnt = result_3.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
total = result_3.collect()[0]['order_count_1995']
assert total > 0, f"Expected order_count_1995 > 0, got {total}"
assert total < 10_000_000, f"order_count_1995 seems unreasonably large: {total}"
print(f"Problem 3 passed ✓  ({cnt} rows, orders in 1995={total})")

## Problem 4

Find the **monthly order volume and average order price** by extracting
year and month from `o_orderdate`. Use `F.year()` and `F.month()` functions.

**Expected output columns:**
- `year` - calendar year
- `month` - calendar month (1–12)
- `order_count` - number of orders in that month
- `avg_price` - average `o_totalprice` for orders in that month

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4 
result_4 = df.groupBy(F.year("o_orderdate").alias("year"),F.month("o_orderdate").alias("month")).agg(F.count("o_orderkey").alias("order_count"),F.avg("o_totalprice").alias("avg_price"))

display(result_4)  # replace this

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'order_count' in cols, "Missing column: order_count"
assert 'avg_price' in cols, "Missing column: avg_price"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
months = [r['month'] for r in result_4.collect()]
assert all(1 <= m <= 12 for m in months), "All month values must be between 1 and 12"
assert all(r['order_count'] > 0 for r in result_4.collect()), "All order counts must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find the **top 5 clerks** by the number of orders they have handled.
Group by `o_clerk` and sort descending.

**Expected output columns:**
- `o_clerk` - clerk identifier
- `order_count` - number of orders handled by that clerk (top 5)

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
result_5 = df.groupBy("o_clerk").agg(F.count("o_orderkey").alias("order_count")).orderBy(F.desc("order_count")).limit(5)

display(result_5)  # replace this

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'o_clerk' in cols, "Missing column: o_clerk"
assert 'order_count' in cols, "Missing column: order_count"
cnt = result_5.count()
assert cnt == 5, f"Expected exactly 5 rows (top 5), got {cnt}"
counts = [r['order_count'] for r in result_5.collect()]
assert counts == sorted(counts, reverse=True), "Results must be sorted by order_count descending"
assert all(c > 0 for c in counts), "All order counts must be positive"
print(f"Problem 5 passed ✓  ({cnt} rows)")